# Embedding geometry experiments

Live measurements that informed the sheaf writeups, run on 2026-08-11 with `all-MiniLM-L6-v2` on CPU. This is a small, deliberately curated corpus — the point is to *observe the geometry* (ranking quality, anisotropy, effective dimensionality, truncation), not to benchmark models.

Sections map onto the docs as follows:
- **1. Ranking** → `embeddings_writeup.html` §4 (Finding 1)
- **2. Anisotropy** → `vector_ratings.html` §4 and `embeddings_writeup.html` §5
- **3. Effective dimensionality** → `embeddings_writeup.html` §3
- **4. Truncation** → `embeddings_writeup.html` §3 (MRL)
- **5. Noise floor** → `embeddings_writeup.html` §4 (Finding 2, chart)

To rerun: `pip install sentence-transformers` (CPU torch is enough); the model downloads to `~/.cache/huggingface` on first use.

## Setup

## 1. Query → problem ranking

Eight unrelated STEM problems; four queries. Embeddings are L2-normalized, so cosine similarity is a plain dot product. Reading the margins: the right problem wins by a wide margin, but the *absolute* scores mean nothing in isolation.

## 2. Anisotropy: embeddings live in a cone, not a sphere

If task directions were uniform over the unit sphere, the mean pairwise cosine between different problems would be ~0. It is not — even on this deliberately *diverse* corpus. Mean-centering the corpus (subtract the mean direction, renormalize) recenters the distribution, which is the fix used for the skill space in `vector_ratings.html` §4.

## 3. Effective dimensionality (PCA)

Tiny sample (8 points → at most 7 nonzero components), but the point stands: nominal dimension ≫ information dimension. Embedding clouds are low-rank; the extra nominal dims are never buying extra skill axes.

## 4. Naive dimension truncation 384 → 64

This model is *not* MRL-trained, yet truncating to the first 64 dims keeps the top-3 ranking here. Don't rely on it in general — MRL-trained models (Qwen3-Embedding, nomic-embed-text-v1.5) guarantee the property and lose ~1% at 256 dims.

## 5. The unrelated-bullshit problem: score distribution

The nonsense query scores *something* — here it peaks at +0.057 on 8 docs, and the relevant query's top score is +0.719. The absolute numbers are not comparable across queries: there is no universal relevance threshold.

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [2]:
problems = [
    ("calc-related-rates", "A ladder 10 ft long leans against a wall. The bottom slides away at 2 ft/s. How fast is the top sliding down when the bottom is 6 ft from the wall?"),
    ("linear-algebra-eig", "Find the eigenvalues and eigenvectors of the matrix [[2, 1], [1, 2]] and diagonalize it."),
    ("probability-bayes", "A disease affects 1 in 1000 people. A test is 99% accurate. If you test positive, what is the probability you have the disease?"),
    ("physics-projectile", "A ball is thrown at 20 m/s at 45 degrees. Find the range, maximum height, and time of flight."),
    ("calc-integral", "Evaluate the integral of x^2 e^x dx by integration by parts."),
    ("number-theory", "Prove that sqrt(2) is irrational. Extend the argument to sqrt(p) for any prime p."),
    ("econ-supply-demand", "Given demand Q = 100 - 2P and supply Q = 3P, find the equilibrium price and the consumer surplus."),
    ("diff-eq", "Solve y'' + 4y = 0 with y(0) = 1, y'(0) = 0, and classify the damping regime."),
]

queries = [
    ("q-ladder", "related rates ladder sliding down a wall"),
    ("q-eig", "how to diagonalize a symmetric matrix"),
    ("q-unrelated", "best recipe for chocolate chip cookies"),
    ("q-vague", "that thing with triangles and speed"),
]

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

names = [n for n, _ in problems]
P = model.encode([t for _, t in problems], normalize_embeddings=True)
Q = model.encode([t for _, t in queries], normalize_embeddings=True)

print("=== query -> problem cosine scores (top 3 each) ===")
for (qn, qt), q in zip(queries, Q):
    sims = P @ q
    order = np.argsort(-sims)[:3]
    print(f"\n{qn}: \"{qt}\" ")
    for i in order:
        print(f"  {sims[i]:+.3f}  {names[i]}")

=== query -> problem cosine scores (top 3 each) ===

q-ladder: "related rates ladder sliding down a wall" 
  +0.719  calc-related-rates
  +0.264  physics-projectile
  +0.112  econ-supply-demand

q-eig: "how to diagonalize a symmetric matrix" 
  +0.584  linear-algebra-eig
  +0.134  number-theory
  +0.057  econ-supply-demand

q-unrelated: "best recipe for chocolate chip cookies" 
  +0.057  calc-integral
  +0.054  linear-algebra-eig
  +0.040  econ-supply-demand

q-vague: "that thing with triangles and speed" 
  +0.276  calc-related-rates
  +0.193  physics-projectile
  +0.057  linear-algebra-eig


In [3]:
print("=== anisotropy ===")
S = P @ P.T
iu = np.triu_indices(len(P), 1)
print(f"mean pairwise cosine between DIFFERENT problems: {S[iu].mean():+.3f}  (min {S[iu].min():+.3f}, max {S[iu].max():+.3f})")
print("if directions were uniform on the sphere this mean would be ~0.00")

mu = P.mean(axis=0)
Pc = P - mu
Pc = Pc / np.linalg.norm(Pc, axis=1, keepdims=True)
Sc = Pc @ Pc.T
print(f"after mean-centering the corpus:               {Sc[iu].mean():+.3f}")

=== anisotropy ===
mean pairwise cosine between DIFFERENT problems: +0.094  (min -0.085, max +0.372)
if directions were uniform on the sphere this mean would be ~0.00
after mean-centering the corpus:               -0.143


In [4]:
print("=== effective dimensionality (PCA on the 8 problems) ===\n")
X = P - P.mean(axis=0)
sv = np.linalg.svd(X, compute_uv=False)
var = sv**2 / (sv**2).sum()
print("explained variance per component:", np.array2string(var, precision=3, suppress_small=True))
print(f"components needed for 90% variance: {np.searchsorted(np.cumsum(var), 0.9) + 1} of {P.shape[1]}")

=== effective dimensionality (PCA on the 8 problems) ===

explained variance per component: [0.193 0.171 0.156 0.141 0.129 0.116 0.094 0.   ]
components needed for 90% variance: 6 of 384


In [5]:
print("=== naive dimension truncation 384 -> 64 (not MRL-trained) ===\n")
q = Q[0]
full = P @ q
P64 = P[:, :64] / np.linalg.norm(P[:, :64], axis=1, keepdims=True)
q64 = q[:64] / np.linalg.norm(q[:64])
tr = P64 @ q64
print(f"q-ladder ranks at 384d: {np.argsort(-full)[:3]} scores {np.sort(-full)[:3]}")
print(f"q-ladder ranks at  64d: {np.argsort(-tr)[:3]} scores {np.sort(-tr)[:3]}")

=== naive dimension truncation 384 -> 64 (not MRL-trained) ===

q-ladder ranks at 384d: [0 3 6] scores [-0.71942884 -0.26427054 -0.11247429]
q-ladder ranks at  64d: [0 3 1] scores [-0.728639   -0.34755695 -0.09831531]


In [6]:
print("=== the 'unrelated bullshit' problem ===\n")
sims = P @ Q[2]
print(f"nonsense query cosines across problems: min {sims.min():+.3f}  max {sims.max():+.3f}")
print(f"relevant q-ladder top score:            {(P @ Q[0]).max():+.3f}")
print("-> bands overlap across queries; no universal threshold exists")

=== the 'unrelated bullshit' problem ===

nonsense query cosines across problems: min -0.057  max +0.057
relevant q-ladder top score:            +0.719
-> bands overlap across queries; no universal threshold exists
